In [6]:
# Bloco 1 — Importar e carregar dados

import pandas as pd
import numpy as np

final_df = pd.read_csv('../outputs/cascade_table.csv')
clustering_df = pd.read_csv('../outputs/clustering_table.csv')

print(f"Shape cascade: {final_df.shape}")
print(f"Shape clustering: {clustering_df.shape}")

Shape cascade: (1388, 9)
Shape clustering: (1388, 16)


In [7]:
#Bloco 2 Montar tabela final 

cols_clustering = ['Item', 'Classe', 'avg_monthly', 'std_monthly', 
                   'ADI', 'CV2', 'D_annual', 'n_clientes', 
                   'n_pedidos', 'demand_class', 'commercial_class']

clustering_slim = clustering_df[cols_clustering].copy()
clustering_slim['Item'] = clustering_slim['Item'].astype(str)
final_df['Item'] = final_df['Item'].astype(str)

simulation_table = final_df.merge(clustering_slim, on='Item', how='left')

cols_order = [
    'Item', 'Classe',
    'demand_class', 'commercial_class', 'policy_cluster',
    'production_level', 'inventory_model',
    'SS', 'ROP', 'EOQ', 's', 'S',
    'avg_monthly', 'std_monthly', 'ADI', 'CV2',
    'D_annual', 'n_clientes', 'n_pedidos',
]

simulation_table = simulation_table[cols_order]

print(f"Total de itens: {len(simulation_table)}")
print(simulation_table['policy_cluster'].value_counts())

Total de itens: 1388
policy_cluster
Dynamic        721
MTO            422
MTS            241
Comakership      4
Name: count, dtype: int64


In [8]:
#Bloco 3  Exportar para o DT

simulation_table.to_csv('../outputs/simulation_table.csv', index=False)

with pd.ExcelWriter('../outputs/simulation_table.xlsx', engine='openpyxl') as writer:
    
    simulation_table.to_excel(writer, sheet_name='All Items', index=False)
    
    for level in simulation_table['production_level'].unique():
        subset = simulation_table[simulation_table['production_level'] == level]
        sheet_name = level.replace('_', ' ').title()[:31]
        subset.to_excel(writer, sheet_name=sheet_name, index=False)
    
    resumo = pd.DataFrame({
        'Cluster':        simulation_table['policy_cluster'].value_counts().index,
        'N_Itens':        simulation_table['policy_cluster'].value_counts().values,
        'SS_medio':       simulation_table.groupby('policy_cluster')['SS'].mean().round(2),
        'ROP_medio':      simulation_table.groupby('policy_cluster')['ROP'].mean().round(2),
        'EOQ_medio':      simulation_table.groupby('policy_cluster')['EOQ'].mean().round(2),
        'D_annual_medio': simulation_table.groupby('policy_cluster')['D_annual'].mean().round(2),
    }).reset_index(drop=True)
    
    resumo.to_excel(writer, sheet_name='Resumo', index=False)

print("Exportação concluída.")

Exportação concluída.
